#JSON
We will be going over JSON fields to help with future projects. <br>
JSON stands for JavaScript Object Notation. It is way of storing and transporting data.


## JSON Syntax
- JSON has six data types:
  - a string
  - a number
  - an object (JSON object)
  - an array
  - a boolean
  - null
- Objects are in key/value pairs and separated by commas
- Curly braces are used for objects
- Square brackets are used for arrays

When intepreted by Python, strings are strings and numbers become either integers or floats.  Objects become dictionaries and arrays become lists.  Booleans are booleans and null is null.

## JSON Example

Name/Value Pair: `'First Name':'Nevin'` <br>
Object: `{'First Name':'Nevin', 'Last Name': 'Martin'}` <br>
Array: `'employees':[{'First Name':'Nevin', 'Last Name': 'Martin'},
{'First Name':'Joe', 'Last Name': 'Olonia'}]`



## Extracting Data from JSON Fields

In [1]:
import pandas as pd
import json
import io


In [2]:
%%capture
%%bash
apt-get update
apt-get install -y jq


In [3]:
!jq --help

jq - commandline JSON processor [version 1.7]

Usage:	jq [options] <jq filter> [file...]
	jq [options] --args <jq filter> [strings...]
	jq [options] --jsonargs <jq filter> [JSON_TEXTS...]

jq is a tool for processing JSON inputs, applying the given filter to
its JSON text inputs and producing the filter's results as JSON on
standard output.

The simplest filter is ., which copies jq's input to its output
unmodified except for formatting. For more advanced filters see
the jq(1) manpage ("man jq") and/or https://jqlang.github.io/jq/.

Example:

	$ echo '{"foo": 0}' | jq .
	{
	  "foo": 0
	}

Command options:
  -n, --null-input          use `null` as the single input value;
  -R, --raw-input           read each line as string instead of JSON;
  -s, --slurp               read all inputs into an array and use it as
                            the single input value;
  -c, --compact-output      compact instead of pretty-printed output;
  -r, --raw-output          output strings without escape

In [4]:
# This function will extract an element within a JSON entry
def extract_json_fields(json_list, field_name):
  '''
  This function can pull an individual item within a JSON field. It takes the following arguments:
  - JSON list in which the item is located
  - The name of the item
  '''
  entry_list = json.loads(json_list)

  # return '|'.join([ str(element[field_name]) for element in entry_list ])
  # return str(entry_list[0][field_name])

  return entry_list[0][field_name]


In [5]:
excel_file = 'https://ddc-datascience.s3.amazonaws.com/animals.xlsx'
excel_file

'https://ddc-datascience.s3.amazonaws.com/animals.xlsx'

In [6]:
# Load the excel sheets into a dictionary
sheets = pd.read_excel( excel_file, sheet_name = None )
sheets.keys()


dict_keys(['Sheet1', 'Sheet2', 'Sheet3'])

In [7]:
animal_dat = sheets["Sheet3"]
animal_dat


,Animal,Info
0,Dog,"[{""id"": 54, ""name"":""Precious""}]"
1,Cat,"[{""id"": 24, ""name"":""Midnight""}]"
2,Cow,"[{""id"": 32, ""name"":""Spots""}]"
3,Mouse,"[{""id"": 58, ""name"":""Fuzzy""}]"


In [8]:
animal_dat.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Animal  4 non-null      object
 1   Info    4 non-null      object
dtypes: object(2)
memory usage: 196.0+ bytes


In [9]:
info_ds = animal_dat['Info'][0]
info_ds

'[{"id": 54, "name":"Precious"}]'

In [10]:
# Notice that the item is a string
type(info_ds)


str

In [11]:
info_ds[0]


'['

In [13]:
# Interpret the string as a JSON object and convert to a Python object
json_ds = json.loads(info_ds)
json_ds


[{'id': 54, 'name': 'Precious'}]

In [14]:
type(json_ds)


list

In [15]:
len(json_ds)

1

In [16]:
json_ds[0]


{'id': 54, 'name': 'Precious'}

In [17]:
type(json_ds[0])


dict

In [18]:
json_ds[0]['id']


54

In [19]:
type(json_ds[0]['id'])


int

In [20]:
json_ds[0].keys()

dict_keys(['id', 'name'])

In [21]:
# Extract the name from the JSON
animal_dat_updated = animal_dat.copy()
animal_dat_updated['Name_Extract'] = animal_dat_updated['Info'].apply(extract_json_fields, field_name = 'name') # extract_json_fields(x, "name")
animal_dat_updated



,Animal,Info,Name_Extract
0,Dog,"[{""id"": 54, ""name"":""Precious""}]",Precious
1,Cat,"[{""id"": 24, ""name"":""Midnight""}]",Midnight
2,Cow,"[{""id"": 32, ""name"":""Spots""}]",Spots
3,Mouse,"[{""id"": 58, ""name"":""Fuzzy""}]",Fuzzy


In [22]:
# Extract the ID from the JSON
animal_dat_updated['ID_Extract']   = animal_dat_updated['Info'].apply(extract_json_fields, field_name = 'id')   # extract_json_fields(x, "id")
animal_dat_updated


,Animal,Info,Name_Extract,ID_Extract
0,Dog,"[{""id"": 54, ""name"":""Precious""}]",Precious,54
1,Cat,"[{""id"": 24, ""name"":""Midnight""}]",Midnight,24
2,Cow,"[{""id"": 32, ""name"":""Spots""}]",Spots,32
3,Mouse,"[{""id"": 58, ""name"":""Fuzzy""}]",Fuzzy,58


In [23]:
animal_dat_updated.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Animal        4 non-null      object
 1   Info          4 non-null      object
 2   Name_Extract  4 non-null      object
 3   ID_Extract    4 non-null      int64 
dtypes: int64(1), object(3)
memory usage: 260.0+ bytes


In [24]:
animal_dat_updated['Info'][0]


'[{"id": 54, "name":"Precious"}]'

In [25]:
foo = animal_dat_updated['Info'][0]
foo

'[{"id": 54, "name":"Precious"}]'

In [26]:
foo = '''[
  {
      "id": 54,
      "name":"Precious"
    }
]'''
print(foo)

[
  {
      "id": 54,
      "name":"Precious"
    }
]


In [27]:
x = json.loads(foo)
x

[{'id': 54, 'name': 'Precious'}]

In [28]:
type(x[0]['id'])

int

In [29]:
animal_dat_updated.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Animal        4 non-null      object
 1   Info          4 non-null      object
 2   Name_Extract  4 non-null      object
 3   ID_Extract    4 non-null      int64 
dtypes: int64(1), object(3)
memory usage: 260.0+ bytes


In [30]:
animal_dat_updated["ID_Extract"][0]

np.int64(54)

In [31]:
animal_json = animal_dat_updated.to_json()
animal_json


'{"Animal":{"0":"Dog","1":"Cat","2":"Cow","3":"Mouse"},"Info":{"0":"[{\\"id\\": 54, \\"name\\":\\"Precious\\"}]","1":"[{\\"id\\": 24, \\"name\\":\\"Midnight\\"}]","2":"[{\\"id\\": 32, \\"name\\":\\"Spots\\"}]","3":"[{\\"id\\": 58, \\"name\\":\\"Fuzzy\\"}]"},"Name_Extract":{"0":"Precious","1":"Midnight","2":"Spots","3":"Fuzzy"},"ID_Extract":{"0":54,"1":24,"2":32,"3":58}}'

In [32]:
print(animal_json)

{"Animal":{"0":"Dog","1":"Cat","2":"Cow","3":"Mouse"},"Info":{"0":"[{\"id\": 54, \"name\":\"Precious\"}]","1":"[{\"id\": 24, \"name\":\"Midnight\"}]","2":"[{\"id\": 32, \"name\":\"Spots\"}]","3":"[{\"id\": 58, \"name\":\"Fuzzy\"}]"},"Name_Extract":{"0":"Precious","1":"Midnight","2":"Spots","3":"Fuzzy"},"ID_Extract":{"0":54,"1":24,"2":32,"3":58}}


In [33]:
type(animal_json)

str

In [34]:
json_animal = json.loads(animal_json)
type(json_animal)


dict

In [35]:
json_animal


{'Animal': {'0': 'Dog', '1': 'Cat', '2': 'Cow', '3': 'Mouse'},
 'Info': {'0': '[{"id": 54, "name":"Precious"}]',
  '1': '[{"id": 24, "name":"Midnight"}]',
  '2': '[{"id": 32, "name":"Spots"}]',
  '3': '[{"id": 58, "name":"Fuzzy"}]'},
 'Name_Extract': {'0': 'Precious',
  '1': 'Midnight',
  '2': 'Spots',
  '3': 'Fuzzy'},
 'ID_Extract': {'0': 54, '1': 24, '2': 32, '3': 58}}

In [36]:
animals_v02 = pd.DataFrame.from_dict(json_animal)
animals_v02



,Animal,Info,Name_Extract,ID_Extract
0,Dog,"[{""id"": 54, ""name"":""Precious""}]",Precious,54
1,Cat,"[{""id"": 24, ""name"":""Midnight""}]",Midnight,24
2,Cow,"[{""id"": 32, ""name"":""Spots""}]",Spots,32
3,Mouse,"[{""id"": 58, ""name"":""Fuzzy""}]",Fuzzy,58


In [37]:
animals_v02.info()


<class 'pandas.core.frame.DataFrame'>
Index: 4 entries, 0 to 3
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Animal        4 non-null      object
 1   Info          4 non-null      object
 2   Name_Extract  4 non-null      object
 3   ID_Extract    4 non-null      int64 
dtypes: int64(1), object(3)
memory usage: 332.0+ bytes


## Reading JSON into a data frame

In [38]:
!jq . sample_data/anscombe.json | head -30

[
  {
    "Series": "I",
    "X": 10.0,
    "Y": 8.04
  },
  {
    "Series": "I",
    "X": 8.0,
    "Y": 6.95
  },
  {
    "Series": "I",
    "X": 13.0,
    "Y": 7.58
  },
  {
    "Series": "I",
    "X": 9.0,
    "Y": 8.81
  },
  {
    "Series": "I",
    "X": 11.0,
    "Y": 8.33
  },
  {
    "Series": "I",
    "X": 14.0,
    "Y": 9.96


In [39]:
foo = json.load( open( 'sample_data/anscombe.json', 'r'))

In [40]:
foo[:30]

[{'Series': 'I', 'X': 10.0, 'Y': 8.04},
 {'Series': 'I', 'X': 8.0, 'Y': 6.95},
 {'Series': 'I', 'X': 13.0, 'Y': 7.58},
 {'Series': 'I', 'X': 9.0, 'Y': 8.81},
 {'Series': 'I', 'X': 11.0, 'Y': 8.33},
 {'Series': 'I', 'X': 14.0, 'Y': 9.96},
 {'Series': 'I', 'X': 6.0, 'Y': 7.24},
 {'Series': 'I', 'X': 4.0, 'Y': 4.26},
 {'Series': 'I', 'X': 12.0, 'Y': 10.84},
 {'Series': 'I', 'X': 7.0, 'Y': 4.81},
 {'Series': 'I', 'X': 5.0, 'Y': 5.68},
 {'Series': 'II', 'X': 10.0, 'Y': 9.14},
 {'Series': 'II', 'X': 8.0, 'Y': 8.14},
 {'Series': 'II', 'X': 13.0, 'Y': 8.74},
 {'Series': 'II', 'X': 9.0, 'Y': 8.77},
 {'Series': 'II', 'X': 11.0, 'Y': 9.26},
 {'Series': 'II', 'X': 14.0, 'Y': 8.1},
 {'Series': 'II', 'X': 6.0, 'Y': 6.13},
 {'Series': 'II', 'X': 4.0, 'Y': 3.1},
 {'Series': 'II', 'X': 12.0, 'Y': 9.13},
 {'Series': 'II', 'X': 7.0, 'Y': 7.26},
 {'Series': 'II', 'X': 5.0, 'Y': 4.74},
 {'Series': 'III', 'X': 10.0, 'Y': 7.46},
 {'Series': 'III', 'X': 8.0, 'Y': 6.77},
 {'Series': 'III', 'X': 13.0, 'Y': 12.7

In [41]:
df = pd.DataFrame( foo )
df

,Series,X,Y
0,I,10.0,8.04
1,I,8.0,6.95
2,I,13.0,7.58
3,I,9.0,8.81
4,I,11.0,8.33
5,I,14.0,9.96
6,I,6.0,7.24
7,I,4.0,4.26
8,I,12.0,10.84
9,I,7.0,4.81


In [42]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44 entries, 0 to 43
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Series  44 non-null     object 
 1   X       44 non-null     float64
 2   Y       44 non-null     float64
dtypes: float64(2), object(1)
memory usage: 1.2+ KB


## Writing a data frame to JSON

In [43]:
df.to_json(path_or_buf="anscombe.json", orient = "table", index= True)

In [44]:
!jq . anscombe.json


{
  "schema": {
    "fields": [
      {
        "name": "index",
        "type": "integer"
      },
      {
        "name": "Series",
        "type": "string"
      },
      {
        "name": "X",
        "type": "number"
      },
      {
        "name": "Y",
        "type": "number"
      }
    ],
    "primaryKey": [
      "index"
    ],
    "pandas_version": "1.4.0"
  },
  "data": [
    {
      "index": 0,
      "Series": "I",
      "X": 10.0,
      "Y": 8.04
    },
    {
      "index": 1,
      "Series": "I",
      "X": 8.0,
      "Y": 6.95
    },
    {
      "index": 2,
      "Series": "I",
      "X": 13.0,
      "Y": 7.58
    },
    {
      "index": 3,
      "Series": "I",
      "X": 9.0,
      "Y": 8.81
    },
    {
      "index": 4,
      "Series": "I",
      "X": 11.0,
      "Y": 8.33
    },
    {
      "index": 5,
      "Series": "I",
      "X": 14.0,
      "Y": 9.96
    },
    {
      "index": 6,
      "Series": "I",
      "X": 6.0,
      "Y": 7.24
    },
    {
      "index":

In [45]:
df = pd.read_json( 'anscombe.json', orient = "table")
df

,Series,X,Y
0,I,10.0,8.04
1,I,8.0,6.95
2,I,13.0,7.58
3,I,9.0,8.81
4,I,11.0,8.33
5,I,14.0,9.96
6,I,6.0,7.24
7,I,4.0,4.26
8,I,12.0,10.84
9,I,7.0,4.81


In [46]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 44 entries, 0 to 43
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Series  44 non-null     object 
 1   X       44 non-null     float64
 2   Y       44 non-null     float64
dtypes: float64(2), object(1)
memory usage: 1.4+ KB


In [47]:
df_dict= json.load( open("anscombe.json", 'r') )
df_small = { "schema": df_dict["schema"],
  "data": df_dict["data"][:10]
}

In [48]:
df_small

{'schema': {'fields': [{'name': 'index', 'type': 'integer'},
   {'name': 'Series', 'type': 'string'},
   {'name': 'X', 'type': 'number'},
   {'name': 'Y', 'type': 'number'}],
  'primaryKey': ['index'],
  'pandas_version': '1.4.0'},
 'data': [{'index': 0, 'Series': 'I', 'X': 10.0, 'Y': 8.04},
  {'index': 1, 'Series': 'I', 'X': 8.0, 'Y': 6.95},
  {'index': 2, 'Series': 'I', 'X': 13.0, 'Y': 7.58},
  {'index': 3, 'Series': 'I', 'X': 9.0, 'Y': 8.81},
  {'index': 4, 'Series': 'I', 'X': 11.0, 'Y': 8.33},
  {'index': 5, 'Series': 'I', 'X': 14.0, 'Y': 9.96},
  {'index': 6, 'Series': 'I', 'X': 6.0, 'Y': 7.24},
  {'index': 7, 'Series': 'I', 'X': 4.0, 'Y': 4.26},
  {'index': 8, 'Series': 'I', 'X': 12.0, 'Y': 10.84},
  {'index': 9, 'Series': 'I', 'X': 7.0, 'Y': 4.81}]}

In [50]:
json.dumps(df_small)

'{"schema": {"fields": [{"name": "index", "type": "integer"}, {"name": "Series", "type": "string"}, {"name": "X", "type": "number"}, {"name": "Y", "type": "number"}], "primaryKey": ["index"], "pandas_version": "1.4.0"}, "data": [{"index": 0, "Series": "I", "X": 10.0, "Y": 8.04}, {"index": 1, "Series": "I", "X": 8.0, "Y": 6.95}, {"index": 2, "Series": "I", "X": 13.0, "Y": 7.58}, {"index": 3, "Series": "I", "X": 9.0, "Y": 8.81}, {"index": 4, "Series": "I", "X": 11.0, "Y": 8.33}, {"index": 5, "Series": "I", "X": 14.0, "Y": 9.96}, {"index": 6, "Series": "I", "X": 6.0, "Y": 7.24}, {"index": 7, "Series": "I", "X": 4.0, "Y": 4.26}, {"index": 8, "Series": "I", "X": 12.0, "Y": 10.84}, {"index": 9, "Series": "I", "X": 7.0, "Y": 4.81}]}'

In [49]:
pd.read_json( json.dumps(df_small), orient="table" )

/tmp/ipykernel_2766/4021369110.py:1: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  pd.read_json( json.dumps(df_small), orient="table" )


,Series,X,Y
0,I,10.0,8.04
1,I,8.0,6.95
2,I,13.0,7.58
3,I,9.0,8.81
4,I,11.0,8.33
5,I,14.0,9.96
6,I,6.0,7.24
7,I,4.0,4.26
8,I,12.0,10.84
9,I,7.0,4.81


In [51]:
pd.read_json(io.StringIO(json.dumps(df_small)), orient="table")


,Series,X,Y
0,I,10.0,8.04
1,I,8.0,6.95
2,I,13.0,7.58
3,I,9.0,8.81
4,I,11.0,8.33
5,I,14.0,9.96
6,I,6.0,7.24
7,I,4.0,4.26
8,I,12.0,10.84
9,I,7.0,4.81
